# Kaggle Music Generator — ACE-Step C++ CUDA job\nOne job per Kaggle CLI run. The request is embedded by GitHub Actions. No GitHub token is used inside Kaggle.

In [ ]:
import os, sys, subprocess, shutil, json\nfrom pathlib import Path\nprint('Python:',sys.version)\nsubprocess.run(['nvidia-smi'],check=True)\nif shutil.which('nvcc') is None: raise RuntimeError('CUDA compiler nvcc is not available.')\nprint(subprocess.check_output(['nvcc','--version'],text=True).splitlines()[-1])

In [ ]:
subprocess.run(['apt-get','update','-qq'],check=True)\nsubprocess.run(['apt-get','install','-y','-qq','build-essential','cmake','ninja-build','git','pkg-config','libcurl4-openssl-dev'],check=True)\nsubprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','hf','six'],check=True)\nsubprocess.run(['hf','--help'],check=True,stdout=subprocess.DEVNULL)

In [ ]:
engine_dir=Path('/kaggle/working/acestep.cpp')\nif not engine_dir.exists(): subprocess.run(['git','clone','--depth','1','--recurse-submodules','https://github.com/ServeurpersoCom/acestep.cpp.git',str(engine_dir)],check=True)\nelse: subprocess.run(['git','submodule','update','--init','--recursive'],cwd=engine_dir,check=True)

In [ ]:
subprocess.run(['cmake','-S','.','-B','build','-DGGML_CUDA=ON','-DCMAKE_CUDA_ARCHITECTURES=75','-DCMAKE_BUILD_TYPE=Release'],cwd=engine_dir,check=True)\nsubprocess.run(['cmake','--build','build','--config','Release','-j2'],cwd=engine_dir,check=True)\nfor b in ('ace-lm','ace-synth'):\n    if not (engine_dir/'build'/b).is_file(): raise RuntimeError('Missing binary: '+b)

In [ ]:
# Q8_0 model set; Kaggle runs a fresh notebook execution per job.\nsubprocess.run(['bash','-lc','./models.sh --lm 4B --quant Q8_0'],cwd=engine_dir,check=True)\nprint('GGUF files:',[p.name for p in (engine_dir/'models').glob('*.gguf')])

In [ ]:
# GitHub Actions replaces this placeholder before kaggle kernels push.\nJOB_PAYLOAD = None\nif JOB_PAYLOAD is None: raise RuntimeError('No job payload was injected.')\nPath('/kaggle/working/job.json').write_text(json.dumps(JOB_PAYLOAD,ensure_ascii=False),encoding='utf-8')\nprint('Running issue #',JOB_PAYLOAD.get('issue_number'))

In [ ]:
repo=Path('/kaggle/working/kaggle-music-generator')\nif not repo.exists(): subprocess.run(['git','clone','--depth','1','https://github.com/unisocsu/kaggle-music-generator.git',str(repo)],check=True)\nsubprocess.run([sys.executable,str(repo/'kaggle'/'worker.py'),'--job-file','/kaggle/working/job.json'],cwd=engine_dir,check=True)\nprint('MP3 saved under /kaggle/working/output/')